In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install \
    torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 \
    --index-url https://download.pytorch.org/whl/cu124
!pip install --upgrade transformers datasets evaluate accelerate speechbrain jiwer

# Imports
import os, random, math, warnings
from pathlib import Path
import itertools
import torch, torchaudio, torchaudio.transforms as T
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset, Dataset
from transformers import Wav2Vec2Model, Wav2Vec2Processor, Wav2Vec2ForCTC
import evaluate
from speechbrain.pretrained import EncoderClassifier

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Config
CFG = {
    "sample_rate"       : 16_000,
    "batch_size"        : 4,
    "num_epochs"        : 3,
    "lr"                : 1e-4,
    "bottleneck_dim"    : 256,
    "dropout"           : 0.1,
    "max_train_hours"   : 10.0,
    "eval_n_examples"   : 128,
}

DATA_ROOT = Path("./librispeech")
DATA_ROOT.mkdir(exist_ok=True, parents=True)

In [ ]:
# Dataset
train_stream = load_dataset(
    "librispeech_asr",
    "clean",
    split="train.100",
    cache_dir=str(DATA_ROOT),
    streaming=True,
)

def take_n_hours(dataset, n_hours):
    out, tot_h = [], 0.0
    for ex in dataset:
        arr = ex["audio"]["array"]
        sr  = ex["audio"]["sampling_rate"]
        dur_h = len(arr) / sr / 3600.0
        out.append(ex)
        tot_h += dur_h
        if tot_h >= n_hours:
            break
    return out

train_samples = take_n_hours(train_stream, CFG["max_train_hours"])

eval_stream = load_dataset(
    "librispeech_asr",
    "clean",
    split="train.100",
    cache_dir=str(DATA_ROOT),
    streaming=True
)


small_eval = list(itertools.islice(eval_stream, CFG["eval_n_examples"]))
eval_ds = Dataset.from_list(small_eval)
eval_ds = eval_ds.shuffle(seed=0)

In [ ]:
# Pretrained model & collate
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")

def collate(batch):
    # raw waveforms
    wavs = [torch.tensor(b["audio"]["array"],dtype=torch.float32) for b in batch]
    # pad to longest
    lengths = [len(w) for w in wavs]
    max_len = max(lengths)
    padded = torch.stack([torch.nn.functional.pad(w, (0, max_len-len(w))) for w in wavs])
    return padded.unsqueeze(1)      # (B,1,T)

train_loader = DataLoader(train_samples, batch_size=CFG["batch_size"],
                          shuffle=True, collate_fn=collate, num_workers=2)

In [ ]:
# Model
import torch.nn.functional as F

class IdentityHider(nn.Module):
    def __init__(self, bottleneck_dim=256, dropout=0.1):
        super().__init__()
        self.enc = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base").to(DEVICE)
        for p in self.enc.parameters():
            p.requires_grad = False
        h = self.enc.config.hidden_size

        self.bn = nn.Sequential(
            nn.Linear(h, bottleneck_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(bottleneck_dim, h),
        )

        stride = math.prod(self.enc.config.conv_stride)  # ≈320
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(h, 512, kernel_size=stride, stride=stride),
            nn.LeakyReLU(0.2),
            nn.Conv1d(512, 256, kernel_size=9, padding=4),
            nn.LeakyReLU(0.2),
            nn.Conv1d(256, 128, kernel_size=9, padding=4),
            nn.LeakyReLU(0.2),
            nn.Conv1d(128, 1, kernel_size=9, padding=4),
            nn.Tanh(),
        )

    def forward(self, wavs):
        x = wavs.squeeze(1)
        z = self.enc(x).last_hidden_state
        z = self.bn(z).permute(0,2,1)
        recon = self.decoder(z)

        T = wavs.size(-1)
        if recon.size(-1) < T:
            recon = F.pad(recon, (0, T - recon.size(-1)))
        else:
            recon = recon[..., :T]
        return recon


In [ ]:
model = IdentityHider(CFG["bottleneck_dim"], CFG["dropout"]).to(DEVICE)
optim = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])

In [ ]:
# Train
model = IdentityHider(CFG["bottleneck_dim"], CFG["dropout"]).to(DEVICE)
optim = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])


wav_L1  = nn.L1Loss()
spec_fn = T.MelSpectrogram(
    sample_rate=CFG["sample_rate"],
    n_fft=400,
    hop_length=160,
    n_mels=64,
).to(DEVICE)
spec_L1 = nn.L1Loss()

for epoch in range(1, CFG["num_epochs"]+1):
    model.train()
    running_loss = 0.0
    for step, wavs in enumerate(train_loader, 1):
        wavs = wavs.to(DEVICE)

        recon = model(wavs)


        loss_wav = wav_L1(recon, wavs)
        spec_orig = spec_fn(wavs.squeeze(1))
        spec_recon = spec_fn(recon.squeeze(1))
        loss_spec = spec_L1(spec_recon, spec_orig)
        loss = loss_wav + loss_spec

        optim.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()

        running_loss += loss.item()
        if step % 100 == 0:
            print(f"Epoch {epoch} | step {step} | Loss {running_loss/step:.4f}")

    torch.cuda.empty_cache()


In [ ]:
# WER
import jiwer

normaliser = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.ExpandCommonEnglishContractions(),
    jiwer.RemovePunctuation(),
    jiwer.Strip(),
    jiwer.RemoveMultipleSpaces()
])

wer_metric = evaluate.load("wer")
asr = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)
asr.eval()


def rms_normalise(x: torch.Tensor, ref: torch.Tensor, eps=1e-8):
    rms_ref = ref.abs().mean(dim=[1,2], keepdim=True)
    rms_x   = x.abs().mean(dim=[1,2], keepdim=True)
    return x * (rms_ref / (rms_x + eps))

def asr_decode(wav_list):
    inputs = processor(
        wav_list,
        sampling_rate=CFG["sample_rate"],
        return_tensors="pt",
        padding=True
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = asr(**inputs).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)

def compute_wer(dataset, n_items=128):
    preds_raw, refs_raw = [], []
    loader = DataLoader(
        dataset.select(range(n_items)),
        batch_size=CFG["batch_size"],
        shuffle=False,
        collate_fn=collate
    )
    idx = 0
    model.eval()
    with torch.no_grad():
        for wavs in loader:
            B = wavs.size(0)
            refs_raw.extend([dataset[idx + i]["text"] for i in range(B)])
            idx += B

            wavs  = wavs.to(DEVICE)
            recon = rms_normalise(model(wavs), wavs)

            recon_list = [r.squeeze(0).cpu().numpy() for r in recon]
            preds_raw.extend(asr_decode(recon_list))


    refs  = [normaliser(r) for r in refs_raw]
    preds = [normaliser(p) for p in preds_raw]

    return wer_metric.compute(predictions=preds, references=refs)
print("Computing WER on small validation slice …")
wer_val = compute_wer(eval_ds, n_items=CFG["eval_n_examples"])
print(f"WER (reconstructed) ≈ {wer_val*100:.2f}%")


In [ ]:
# Speaker Similarity
from torch.nn.functional import cosine_similarity

embedder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": DEVICE}
)
embedder.eval()

def get_embedding(wav_np):
    wav = torch.tensor(wav_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = embedder.encode_batch(wav)
    return emb.squeeze(0).cpu()


anchors = {}
for ex in eval_ds:
    spk = ex["speaker_id"]
    if spk not in anchors:
        anchors[spk] = get_embedding(ex["audio"]["array"])
    if len(anchors) >= 50:
        break

def similarity_drop(n_items=128):
    base_sims, anon_sims = [], []
    for ex in eval_ds.select(range(n_items)):
        spk = ex["speaker_id"]
        arr = ex["audio"]["array"]


        orig_emb = get_embedding(arr)
        base_sims.append(
            cosine_similarity(orig_emb.squeeze(), anchors[spk].squeeze(), dim=0).item()
        )


        wav = torch.tensor(arr, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
        recon = model(wav)[0, 0].detach().cpu().numpy()
        recon_emb = get_embedding(recon)
        anon_sims.append(
            cosine_similarity(recon_emb.squeeze(), anchors[spk].squeeze(), dim=0).item()
        )

    base = torch.tensor(base_sims)
    anon = torch.tensor(anon_sims)
    return (1 - anon.mean() / base.mean()).item() * 100


print("Measuring similarity drop …")
drop_pct = similarity_drop(n_items=CFG["eval_n_examples"])
print(f"Average speaker-similarity reduction ≈ {drop_pct:.1f}%")


In [ ]:
# result
torch.save(model.state_dict(), "latest_3.pth")

print("\n──────── Summary ────────")
print(f"Final WER      : {wer_val*100:.2f}%")
print(f"Speaker sim ↓  : {drop_pct:.1f}%")
print("Weights saved to identity_hider.pt")



In [ ]:

import torch
import librosa
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
from jiwer import wer




processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-960h")
model     = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-960h").to(DEVICE)


def transcribe(path):
    wav, sr = librosa.load(path, sr=16_000)
    input_values = processor(wav, return_tensors="pt", sampling_rate=sr).input_values.to(DEVICE)
    logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0].lower()


In [ ]:

sample = eval_ds[0]
print(sample.keys())
print(sample["audio"])


In [ ]:

sample = eval_ds[0]
print(sample.keys())
print(sample["speaker_id"])


In [ ]:
# Original WER
def transcribe_audio(wav_array, sr=16000):
    input_values = processor(wav_array, return_tensors="pt", sampling_rate=sr).input_values.to(DEVICE)
    logits = model(input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0].lower()

refs, hyps = [], []

for sample in eval_ds:
    wav_array = sample["audio"]["array"]
    sr        = sample["audio"]["sampling_rate"]
    hyps.append(transcribe_audio(wav_array, sr))
    refs.append(sample["text"].lower())

original_wer = wer(refs, hyps) * 100
print(f"Original WER: {original_wer:.2f}%")


In [ ]:
# Original similarity
import torch
from torch.nn.functional import cosine_similarity


N = CFG["eval_n_examples"]
base_sims = []
for ex in eval_ds.select(range(N)):
    spk = ex["speaker_id"]
    arr = ex["audio"]["array"]
    orig_emb = get_embedding(arr)
    sim = cosine_similarity(orig_emb.squeeze(), anchors[spk].squeeze(), dim=0).item()
    base_sims.append(sim)

orig_mean_sim = torch.tensor(base_sims).mean().item() * 100
print(f"Average ORIGINAL speaker‐similarity ≈ {orig_mean_sim:.1f}%")


In [ ]:
# DEBUG
probe_loader = DataLoader(
    eval_ds.select(range(3)),
    batch_size=1,
    shuffle=False,
    collate_fn=collate
)

model.eval()
for idx, wavs in enumerate(probe_loader, 1):
    wavs  = wavs.to(DEVICE)
    recon = model(wavs)

    # pull out the 1-D arrays
    orig_np  = wavs[0, 0].detach().cpu().numpy()
    recon_np = recon[0, 0].detach().cpu().numpy()

    print(f"\n─── Sample {idx} ───")
    print("GT text   :", eval_ds[idx-1]["text"])
    print("ASR orig  :", asr_decode([orig_np])[0])
    print("ASR recon :", asr_decode([recon_np])[0])


In [ ]:
# graph 1
import matplotlib.pyplot as plt

labels = ["Original WER", "Anonymized WER"]
values = [original_wer, wer_val * 100]

plt.figure()
plt.bar(labels, values)
plt.ylabel("WER (%)")
plt.title("Original vs. Anonymized WER")
plt.ylim(0, max(values) * 1.2)
plt.show()


In [ ]:
# model diagram
import matplotlib.pyplot as plt
import matplotlib.patches as patches


fig, ax = plt.subplots(figsize=(8, 3))

encoder_box = patches.Rectangle((0.05, 0.3), 0.25, 0.4, edgecolor='black', facecolor='lightgray')
bn_box = patches.Rectangle((0.4, 0.35), 0.2, 0.3, edgecolor='black', facecolor='lightblue')
decoder_box = patches.Rectangle((0.7, 0.3), 0.25, 0.4, edgecolor='black', facecolor='lightgreen')

for box in [encoder_box, bn_box, decoder_box]:
    ax.add_patch(box)


ax.annotate('', xy=(0.3, 0.5), xytext=(0.4, 0.5), arrowprops=dict(arrowstyle='<-'))
ax.annotate('', xy=(0.6, 0.5), xytext=(0.7, 0.5), arrowprops=dict(arrowstyle='<-'))


ax.text(0.175, 0.65, 'Encoder\n(Wav2Vec2Model)', ha='center', va='center', fontsize=10)
ax.text(0.5, 0.5, 'Bottleneck\nLinear → ReLU\nDropout → Linear', ha='center', va='center', fontsize=10)
ax.text(0.825, 0.65, 'Decoder\nConvTranspose1d → LeakyReLU\nConv1d layers\nTanh', ha='center', va='center', fontsize=10)


ax.annotate('Raw audio\n(wavs)', xy=(0.05, 0.5), xytext=(0.0, 0.5),
            arrowprops=dict(arrowstyle='->'), ha='right', va='center', fontsize=9)
ax.annotate('Reconstructed\naudio', xy=(1.0, 0.5), xytext=(0.95, 0.5),
            arrowprops=dict(arrowstyle='<-'), ha='left', va='center', fontsize=9)
ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# graph 2
import matplotlib.pyplot as plt


losses = [
    1.5179, 1.4823, 1.4567, 1.4600, 1.4565, 1.4402,
    1.3317, 1.3323, 1.3386, 1.3262, 1.3202, 1.3129,
    1.2278, 1.2286, 1.2593, 1.2772, 1.2712, 1.2768
]

epochs = list(range(1, len(losses) + 1))

plt.figure()
plt.plot(epochs, losses, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss Curve")
plt.xticks(epochs)
plt.tight_layout()
plt.show()


In [ ]:
# graph 3 4
import matplotlib.pyplot as plt

original_wer = 0.79
reconstructed_wer = 14.85

original_sim = 78.1

reduction_pct = 85.4
anonymized_sim = original_sim * (1 - reduction_pct / 100)

plt.figure()
plt.bar(["Original WER", "Reconstructed WER"], [original_wer, reconstructed_wer])
plt.ylabel("WER (%)")
plt.ylim(0, 20)
plt.title("Original vs. Reconstructed WER")
plt.tight_layout()
plt.show()

plt.figure()
plt.bar(["Original Similarity", "Anonymized Similarity"], [original_sim, anonymized_sim])
plt.ylabel("Cosine Similarity (%)")
plt.ylim(0, 100)
plt.title("Speaker Similarity Before vs. After")
plt.tight_layout()
plt.show()


In [ ]:
# spectorgram examples
import torch
import matplotlib.pyplot as plt
import torchaudio

spec_fn = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG["sample_rate"],
    n_fft=400,
    hop_length=160,
    n_mels=64
).to(DEVICE)
db_fn = torchaudio.transforms.AmplitudeToDB().to(DEVICE)


N = 2 # no. of examples

for i in range(N):

    wav_np = eval_ds[i]["audio"]["array"]
    wav = torch.tensor(wav_np, dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        recon = model(wav.unsqueeze(1))[0,0]


    spec_orig = db_fn(spec_fn(wav))
    spec_recon = db_fn(spec_fn(recon.unsqueeze(0)))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    im1 = ax1.imshow(spec_orig[0].cpu(), aspect="auto", origin="lower")
    ax1.set_title(f"Original Spectrogram #{i+1}")
    ax1.set_xlabel("Time Frames")
    ax1.set_ylabel("Mel Bins")
    im2 = ax2.imshow(spec_recon[0].cpu(), aspect="auto", origin="lower")
    ax2.set_title(f"Reconstructed Spectrogram #{i+1}")
    ax2.set_xlabel("Time Frames")
    ax2.set_ylabel("Mel Bins")

    plt.tight_layout()
    plt.show()
